In [ ]:
import jax
import jax.numpy as jnp
import jax_cfd.base as cfd
import h5py
import matplotlib.pyplot as plt

In [ ]:
# import jax_cfd.base as cfd
# help(cfd.advection.advect_van_leer)

In [ ]:


# --- 1. CONFIGURATION ---
# --- Stability Optimized Config ---
size = 64
num_steps = 1000       # More steps to cover the same physical time
viscosity = 0.002      # A "sweet spot" for 64x64 grid
dt = 0.001            # Reduced 10x from your previous run to prevent NaNs
maximum_velocity = 7.0 # High enough for shocks, low enough for stability


# --- Updated Config for Sharp Shocks ---
# num_steps = 200      # Increase steps so we see the evolution longer
# viscosity = 0.002    # Drop this significantly (from 0.05)
# dt = 0.005           # Smaller time step for stability with lower viscosity
# maximum_velocity = 10.0 # Higher speed to force the non-linear "shocking"
peak_wavenumber = 2.0   # Larger initial blobs

# size = 64
num_sims = 100

# num_steps = 50
# viscosity = 0.05
# dt = 0.01

grid = cfd.grids.Grid(shape=(size, size), domain=((0, 2 * jnp.pi), (0, 2 * jnp.pi)))

# --- 2. PHYSICS STEP (SINGLE) ---
def single_step(v_array):
    periodic_bc = cfd.boundaries.periodic_boundary_conditions(ndim=2)
    
    # Wrap with correct C-grid offsets
    u_var = cfd.grids.GridVariable(
        cfd.grids.GridArray(v_array[0], offset=(0.5, 0), grid=grid), 
        bc=periodic_bc
    )
    v_var = cfd.grids.GridVariable(
        cfd.grids.GridArray(v_array[1], offset=(0, 0.5), grid=grid), 
        bc=periodic_bc
    )
    v_tuple = (u_var, v_var)
    
    # Advection
    u_adv = cfd.advection.advect_van_leer(u_var, v_tuple, dt)
    v_adv = cfd.advection.advect_van_leer(v_var, v_tuple, dt)
    
    # Diffusion
    u_next = cfd.diffusion.diffuse(cfd.grids.GridVariable(u_adv, periodic_bc), viscosity * dt)
    v_next = cfd.diffusion.diffuse(cfd.grids.GridVariable(v_adv, periodic_bc), viscosity * dt)
    
    # Return data
    return jnp.stack([u_next.data, v_next.data])

# --- 3. TRAJECTORY & BATCHING ---
def generate_trajectory(initial_state):
    """Uses lax.scan to compute the entire time-series on GPU."""
    def scan_step(carry, _):
        next_val = single_step(carry)
        return next_val, next_val
    
    _, trajectory = jax.lax.scan(scan_step, initial_state, jnp.arange(num_steps))
    return trajectory

# Vectorize (vmap) and compile (jit) the generator
batch_gen_jit = jax.jit(jax.vmap(generate_trajectory))

# --- 4. INITIAL CONDITIONS ---
def get_initial_batch(num_samples):
    keys = jax.random.split(jax.random.PRNGKey(42), num_samples)
    
    def create_single_v0(key):
        # Increased maximum_velocity and decreased peak_wavenumber
        v0 = cfd.initial_conditions.filtered_velocity_field(
            key, grid, maximum_velocity=maximum_velocity, peak_wavenumber=peak_wavenumber
        )
        return jnp.stack([v0[0].data, v0[1].data])
    
    return jax.vmap(create_single_v0)(keys)

# --- 5. EXECUTION ---
print(f"Generating {num_sims} simulations...")
initial_batch = get_initial_batch(num_sims)
# Result shape: [Batch, Time, Channel, X, Y]
all_data = batch_gen_jit(initial_batch)
print("Generation complete. Shape:", all_data.shape)



In [ ]:
# --- 6. STORAGE (HDF5) ---
filename = "burgers_2d_dataset.h5"
with h5py.File(filename, 'w') as f:
    dset = f.create_dataset("sim_data", data=all_data, compression="gzip")
    dset.attrs['description'] = "2D Burgers' Equation Dataset (Simplified The Well)"
    dset.attrs['viscosity'] = viscosity
    dset.attrs['dt'] = dt
print(f"Data saved to {filename}")



In [ ]:
# --- 7. VISUALIZATION (Fixed Scale) ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sample_idx = 0
timesteps_to_plot = [0, 1, 10]

# Find the global max/min for a consistent scale
v_min = jnp.min(all_data[sample_idx, :, 0])
v_max = jnp.max(all_data[sample_idx, :, 0])

for i, t in enumerate(timesteps_to_plot):
    im = axes[i].imshow(all_data[sample_idx, t, 0], 
                       cmap='magma', 
                       vmin=v_min, vmax=v_max) # Fixed scale!
    axes[i].set_title(f"Time Step {t}")
    plt.colorbar(im, ax=axes[i])

plt.suptitle("2D Burgers' Evolution: If this works, you'll see sharp 'cracks' form.")
plt.show()

